A major airline operating in New York City sought to better understand its flight operations by analyzing patterns in flight schedules and durations. The goal of this analysis was to explore how different airlines and routes perform in terms of flight frequency and travel time, with the aim of improving operational efficiency and the passenger experience.

Using the nycflights2022 dataset from the ModernDive team, I conducted an exploratory data analysis of flights departing from JFK, LaGuardia (LGA), and Newark (EWR) airports in the second half of 2022. The dataset included detailed information on departure and arrival times, destinations, carriers, and flight durations.

The analysis focused on identifying the busiest airline-route combinations, calculating average flight times, and summarizing key operational patterns across NYC airports:

------------------------------------------------------

`flights2022-h2.csv` contains information about each flight, including:

Variables:

`carrier`: Airline carrier code

`origin`: Origin airport (IATA code)

`dest`: Destination airport (IATA code)

`air_time`: Duration of the flight in air, in minutes

------------------------------------------------------

`airlines.csv` contains information about each airline:

Variables:

`carrier`: Airline carrier code

`name`: Full name of the airline

------------------------------------------------------

`airports.csv` provides details of airports:

Variables:

`faa`: FAA code of the airport

`name`: Full name of the airport

In [31]:
# Import required packages
library(dplyr)
library(readr)

# Supress Messages
options(readr.show_col_types=FALSE)

# Load the data
flights <- read_csv("flights2022-h2.csv")
airlines <- read_csv("airlines.csv")
airports <- read_csv("airports.csv")

In [32]:
# Which airline and airport pair receives the most flights from NYC and what is the average duration of that flight?
travel <- left_join(flights, airlines, 
					by='carrier')

travel <- left_join(travel, airports, 
					by= c('dest'='faa')) %>%
		  rename('airline_name' = 'name.x',
				 'airport_name' = 'name.y')

frequent <- suppressMessages(travel %>%
		group_by(airline_name, airport_name) %>%
        summarise(flight_count=n(),
				  avg_duration = mean(air_time, na.rm=TRUE)) %>%
		ungroup() %>%
		arrange(desc(flight_count)) %>%
        slice(1) %>%
        select(airline_name, airport_name))
frequent

airline_name,airport_name
<chr>,<chr>
Delta Air Lines Inc.,Hartsfield Jackson Atlanta International Airport


In [33]:
# Find the airport that has the longest average flight duration (in hours) from NYC. What is the name of this airport?
travel <- travel %>%
				 mutate(air_hours = air_time/60)

longest <- suppressMessages(travel %>%
	   group_by(airport_name, airline_name) %>%
	   summarise(avg_duration_hrs = mean(air_hours, na.rm = TRUE)) %>%
	   ungroup %>%
	   arrange(desc(avg_duration_hrs)) %>%
	   slice(1))
longest

airport_name,airline_name,avg_duration_hrs
<chr>,<chr>,<dbl>
Daniel K Inouye International Airport,Delta Air Lines Inc.,10.71667


In [34]:
# Which airport is the least frequented destination for flights departing from JFK?
jfk_departs <- suppressMessages(travel %>%
		filter(origin == 'JFK') %>%
		group_by(origin, destination = airport_name) %>%
		summarise(travel_frequency = n()) %>%
        arrange(travel_frequency))

least <- suppressMessages(jfk_departs %>%
					 pull(destination) %>%
					 first() %>%
					 as.character())
least

[1] "Eagle County Regional Airport"